In [8]:
from dataclasses import dataclass
from typing import Literal
import pandas as pd 

from sun_rad_heating_functions import data_combined, a_d_8, calculate_sun_rad_heat

In [9]:
solar_intensity_base_56 = pd.DataFrame(data_combined)

In [10]:
@dataclass
class Window():
    area_sqm: float
    orientation: Literal['nw', 'n', 'ne', 'e', 'se', 's', 'sw', 'w']
    cooling_load: pd.DataFrame = None
    R_value_m2K_W: float = 1.45
    K1: float = 0.9 
    K2: float = 1 
    K3: float = 1 
    K4_SF_solar_factor: float = 0.34

@dataclass
class Location():
    city_name: str = "Москва"
    latitude: int = 56
    t_out_day_mid: float = 23 # Средняя температура воздуха дневная
    A_air_day: float = 24 # Амплитуда максимальная температуры воздуха

@dataclass
class Room():
    number: str
    name: str
    windows: list[Window]
    cooling_load: pd.DataFrame = None
    required_air_temperature_C: int = 24

@dataclass
class Apartment():
    number: str 
    rooms: list[Room]
    cooling_load: pd.DataFrame = None

@dataclass
class Level():
    number: str
    elevation_m: float 
    apartment: list[Apartment]
    cooling_load: pd.DataFrame = None

@dataclass
class Building():
    number: str
    name: str
    levels: list[Level]
    location: Location
    cooling_load: pd.DataFrame = None




In [11]:
building_location = Location(
    city_name = "Москва",
    latitude = 56,
    t_out_day_mid = 23,
    A_air_day = 24,
)

window_1 = Window(
    area_sqm = 3,
    orientation = "E",
    R_value_m2K_W = 1.45,
    K3 = 1,
    K4_SF_solar_factor = 0.34,
)
window_2 = Window(
    area_sqm = 3,
    orientation = "S",
    R_value_m2K_W = 1.45,
    K3 = 1,
    K4_SF_solar_factor = 0.34
)

room_1 = Room(
    number = '1',
    name = 'bedroom',
    windows = [window_1, window_2],
    required_air_temperature_C = 24
)

room_2 = Room(
    number = '2',
    name = 'kitchen',
    windows = [window_2],
    required_air_temperature_C = 24,
)

apartment_21 = Apartment(
    number='23',
    rooms=[room_1, room_2],
)

level_2 = Level(
    number = '02',
    elevation_m = 5.0, 
    apartment = [apartment_21]
)

building_A = Building(
    number= "1",
    name= "building_5",
    levels= [level_2],
    location= building_location,
)

In [12]:
class Calculation_solar_rad_heat_kW():
    def __init__(self,
                 df_sun_base: pd.DataFrame,
                 building: Building, 
                 location: Location,
                 ):
        self.df_sun_base = df_sun_base,
        self.building = building,
        self.levels = building.levels,
        self.location = location,

        self.apartments = []
        for level in building.levels:
            for apartment in level.apartment:
                self.apartments.append(apartment)

        self.rooms = []
        for apartment in self.apartments:
            for room in apartment.rooms:   
                self.rooms.append(room)

        self.windows = []
        for room in self.rooms:
            for window in room.windows:   
                self.windows.append(window)


        #calculate solar heat tghrough window 
        result_w = None
        for window in self.windows: 
            if result_w is None:
                result_w = calculate_sun_rad_heat(
                    result_sun_base = self.df_sun_base, 
                    A_m2= window.area_sqm,
                    orient = window.orientation,
                    K1= 1.0,
                    K2= 0.9,
                    K3= window.K3,
                    K4= window.K4_SF_solar_factor,
                )




In [13]:
b_A = Calculation_solar_rad_heat_kW(
    df_sun_base=solar_intensity_base_56,
    building=building_A,
    location=building_location,
)

AttributeError: 'tuple' object has no attribute 'copy'

In [ ]:
# print(b_A.building)
# print(b_A.apartments)
# print(b_A.rooms)
print(b_A.windows)

[Window(area_sqm=3, orientation='E', cooling_load=None, R_value_m2K_W=1.45, K1=0.9, K2=1, K3=1, K4_SF_solar_factor=0.34), Window(area_sqm=3, orientation='S', cooling_load=None, R_value_m2K_W=1.45, K1=0.9, K2=1, K3=1, K4_SF_solar_factor=0.34), Window(area_sqm=3, orientation='S', cooling_load=None, R_value_m2K_W=1.45, K1=0.9, K2=1, K3=1, K4_SF_solar_factor=0.34)]


In [ ]:
b_A.windows[0]

Window(area_sqm=3, orientation='E', cooling_load=None, R_value_m2K_W=1.45, K1=0.9, K2=1, K3=1, K4_SF_solar_factor=0.34)